# 模型误差与预测分析

直接使用已保存的 Test 预测；若缺少结果则自动补齐。


In [ ]:
from experiment_pipeline import Pipeline
run = Pipeline()
run.comparison()
run.random_forest()  # 11.1 还需要树模型的原始预测
globals().update(run.lightgbm())


# 9. 模型误差分析

## 9.1 为什么 MAE 与 RMSE 的模型排序相反？

实验 A 比较 Test 上的 `p_target` 与 HLR、N-HLR、RF-p (E2)、LGBM-p (E4) 的预测分布；实验 B 按真实回忆率分解 MAE 与 MSE 的全局贡献；最后检查误差尾部。**全部数值统计使用完整 Test**，只复用第 8 节已算出的 `test_predictions`，不重新训练。

注意评估标签是裁剪后的 `p_target∈[MIN_P,MAX_P]`，因此边界组写作 `p_target=MIN_P` 和 `p_target=MAX_P`，不写作严格的 `p=0/1`。分布图用相同分箱；上边界的大量样本另由统计表列出。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

task92_needed = ["test_true_p", "test_predictions", "MIN_P", "MAX_P"]
task92_missing = [name for name in task92_needed if name not in globals()]
if task92_missing:
    raise RuntimeError(
        "Section 9.1 needs the current kernel's section 8 predictions: "
        + ", ".join(task92_missing)
    )

task92_models = ["HLR", "N-HLR", "RF-p (E2)", "LGBM-p (E4)"]
task92_true = np.asarray(test_true_p, dtype=np.float64)
task92_n = len(task92_true)
task92_series = {"p_target (true)": task92_true}
for name in task92_models:
    task92_series[name] = np.asarray(test_predictions[name][0], dtype=np.float64)
assert all(len(values) == task92_n and np.isfinite(values).all()
           for values in task92_series.values())
assert all(np.all((MIN_P <= values) & (values <= MAX_P))
           for values in task92_series.values())

task92_distribution_rows = []
task92_edges = np.linspace(MIN_P, MAX_P, 51)
task92_histograms = {}
for name, values in task92_series.items():
    p10, p50, p90 = np.quantile(values, [0.10, 0.50, 0.90])
    task92_distribution_rows.append({
        "series": name, "n_test": task92_n,
        "mean": float(values.mean()), "std": float(values.std()),
        "P10": float(p10), "P50": float(p50), "P90": float(p90),
        "share_ge_0.95": float(np.mean(values >= 0.95)),
        "share_at_MIN_P": float(np.mean(values == MIN_P)),
        "share_at_MAX_P": float(np.mean(values == MAX_P)),
    })
    task92_histograms[name] = np.histogram(values, bins=task92_edges)[0] / task92_n

task92_distribution = pd.DataFrame(task92_distribution_rows)
display(task92_distribution.round(6))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
colors = ["black", "#4C78A8", "#54A24B", "#F58518", "#E45756"]
for (name, masses), color in zip(task92_histograms.items(), colors):
    for ax in axes:
        ax.stairs(masses, task92_edges, label=name, linewidth=1.7, color=color)
axes[0].set(
    title="Full Test distribution", xlim=(0, 1),
    xlabel="p_target or predicted p", ylabel="share of all Test rows / bin",
)
axes[0].set_yscale("log")
axes[1].set(
    title="Interior zoom (p < 0.95)", xlim=(0, 0.95),
    xlabel="p_target or predicted p", ylabel="share of all Test rows / bin",
)
for ax in axes:
    ax.grid(alpha=0.2)
axes[1].legend(loc="upper left", fontsize=8)
fig.tight_layout()
plt.show()


### 实验 B：按真实 `p_target` 分解误差

对每组同时报告组内 `MAE_g`、`MSE_g` 和它们对**全局**指标的贡献：`C_MAE=(N_g/N)×MAE_g`、`C_MSE=(N_g/N)×MSE_g`。各组 `C_MAE` 之和应等于整体 MAE；各组 `C_MSE` 之和应等于整体 RMSE²。重点比较表中的差值定义为 **HLR − LGBM-p**：正数表示该组让 HLR 的全局误差更大。

In [ ]:
task92_group_names = [
    "p_target=MIN_P", "MIN_P<p<=0.2", "0.2<p<=0.5",
    "0.5<p<MAX_P", "p_target=MAX_P",
]
task92_group_masks = [
    task92_true == MIN_P,
    (task92_true > MIN_P) & (task92_true <= 0.2),
    (task92_true > 0.2) & (task92_true <= 0.5),
    (task92_true > 0.5) & (task92_true < MAX_P),
    task92_true == MAX_P,
]
task92_covered = np.zeros(task92_n, dtype=np.int8)
for mask in task92_group_masks:
    task92_covered += mask
assert np.all(task92_covered == 1), "p_target groups must cover Test exactly once"

task92_decomposition_rows = []
task92_check_rows = []
for model in task92_models:
    error = task92_series[model] - task92_true
    abs_error = np.abs(error)
    squared_error = error ** 2
    for group, mask in zip(task92_group_names, task92_group_masks):
        n_group = int(mask.sum())
        if not n_group:
            raise ValueError(f"Empty p_target group: {group}")
        task92_decomposition_rows.append({
            "group": group, "n_group": n_group,
            "share": n_group / task92_n, "model": model,
            "MAE_g": float(abs_error[mask].mean()),
            "MSE_g": float(squared_error[mask].mean()),
            "C_MAE": float(abs_error[mask].sum() / task92_n),
            "C_MSE": float(squared_error[mask].sum() / task92_n),
        })
    rows = [row for row in task92_decomposition_rows if row["model"] == model]
    mae = float(abs_error.mean())
    mse = float(squared_error.mean())
    sum_mae = sum(row["C_MAE"] for row in rows)
    sum_mse = sum(row["C_MSE"] for row in rows)
    assert np.isclose(mae, sum_mae, rtol=0, atol=1e-12)
    assert np.isclose(mse, sum_mse, rtol=0, atol=1e-12)
    task92_check_rows.append({
        "model": model, "overall_MAE": mae, "sum_C_MAE": sum_mae,
        "overall_RMSE": float(np.sqrt(mse)),
        "sqrt_sum_C_MSE": float(np.sqrt(sum_mse)),
    })

task92_decomposition = pd.DataFrame(task92_decomposition_rows)
task92_checks = pd.DataFrame(task92_check_rows)
display(task92_decomposition.round(6))
display(task92_checks.round(6))

task92_main_rows = []
for group in task92_group_names:
    hlr = task92_decomposition.loc[
        (task92_decomposition["group"] == group)
        & (task92_decomposition["model"] == "HLR")
    ].iloc[0]
    e4 = task92_decomposition.loc[
        (task92_decomposition["group"] == group)
        & (task92_decomposition["model"] == "LGBM-p (E4)")
    ].iloc[0]
    task92_main_rows.append({
        "group": group, "n_group": int(hlr["n_group"]),
        "share": float(hlr["share"]),
        "HLR_MAE_g": float(hlr["MAE_g"]),
        "E4_MAE_g": float(e4["MAE_g"]),
        "HLR_C_MAE": float(hlr["C_MAE"]),
        "E4_C_MAE": float(e4["C_MAE"]),
        "delta_C_MAE_HLR_minus_E4": float(hlr["C_MAE"] - e4["C_MAE"]),
        "HLR_C_MSE": float(hlr["C_MSE"]),
        "E4_C_MSE": float(e4["C_MSE"]),
        "delta_C_MSE_HLR_minus_E4": float(hlr["C_MSE"] - e4["C_MSE"]),
    })
task92_main = pd.DataFrame(task92_main_rows)
display(task92_main.round(6))

x = np.arange(len(task92_group_names))
width = 0.36
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for ax, metric, title in zip(
    axes, ["C_MAE", "C_MSE"],
    ["Contribution to overall MAE", "Contribution to overall MSE"],
):
    hlr_values = task92_decomposition.loc[
        task92_decomposition["model"] == "HLR", metric
    ].to_numpy()
    e4_values = task92_decomposition.loc[
        task92_decomposition["model"] == "LGBM-p (E4)", metric
    ].to_numpy()
    ax.bar(x - width / 2, hlr_values, width, label="HLR", color="#4C78A8")
    ax.bar(x + width / 2, e4_values, width,
           label="LGBM-p (E4)", color="#E45756")
    ax.set(title=title, xticks=x,
           xticklabels=["MIN_P", "(MIN,.2]", "(.2,.5]", "(.5,MAX)", "MAX_P"])
    ax.grid(axis="y", alpha=0.2)
    ax.legend()
fig.tight_layout()
plt.show()


### 误差尾部：少数大误差是否主导 MSE？

分别计算每个模型 `|error|` 的 P90/P99，以及该模型**自身误差最大 1% 行**对其总 MSE 的贡献。再固定 **HLR 误差最大 1% 的同一批行**，比较四个模型在这些行上的误差；这个固定子集能直接检验 Direct-p 是否改善 HLR 最难的样本。

In [ ]:
task92_top_k = int(np.ceil(task92_n * 0.01))
task92_tail_rows = []
for model in task92_models:
    error = task92_series[model] - task92_true
    abs_error = np.abs(error)
    squared_error = error ** 2
    p90, p99 = np.quantile(abs_error, [0.90, 0.99])
    top_idx = np.argpartition(squared_error, task92_n - task92_top_k)[
        task92_n - task92_top_k:
    ]
    task92_tail_rows.append({
        "model": model, "top_k": task92_top_k,
        "P90_abs_error": float(p90), "P99_abs_error": float(p99),
        "own_worst_1pct_share_of_total_MSE": float(
            squared_error[top_idx].sum() / squared_error.sum()
        ),
    })
task92_tail = pd.DataFrame(task92_tail_rows)
display(task92_tail.round(6))

hlr_abs_error = np.abs(task92_series["HLR"] - task92_true)
hlr_worst_idx = np.argpartition(hlr_abs_error, task92_n - task92_top_k)[
    task92_n - task92_top_k:
]
task92_fixed_rows = []
for model in task92_models:
    error = task92_series[model] - task92_true
    squared_error = error ** 2
    task92_fixed_rows.append({
        "model": model, "fixed_HLR_worst_rows": task92_top_k,
        "MAE_on_rows": float(np.abs(error[hlr_worst_idx]).mean()),
        "MSE_on_rows": float(squared_error[hlr_worst_idx].mean()),
        "share_of_model_total_MSE": float(
            squared_error[hlr_worst_idx].sum() / squared_error.sum()
        ),
    })
task92_fixed_hlr_worst = pd.DataFrame(task92_fixed_rows)
display(task92_fixed_hlr_worst.round(6))

task92_worst_group = pd.DataFrame([
    {
        "group": name,
        "rows_in_HLR_worst_1pct": int(mask[hlr_worst_idx].sum()),
        "share_within_HLR_worst_1pct": float(mask[hlr_worst_idx].mean()),
    }
    for name, mask in zip(task92_group_names, task92_group_masks)
])
display(task92_worst_group.round(6))


# 11. 拓展分析

以下分析检查树模型的逐样本输出，以及概率校准和遗忘风险排序。


## 11.1 RF 与 LightGBM：逐样本预测是否相同

比较同一批 Test 行的 E1（RF-logh）与 E3（LGBM-logh），以及 E2（RF-p）与 E4（LGBM-p）。裁剪后的 `p_hat` 直接使用第 8 节的 `test_predictions`；裁剪前输出通过已训练模型分批重新预测，**不重新训练**。E1/E3 的原始输出为 `log2(h)`，E2/E4 的原始输出为 `p`，两种原始尺度不要互相比大小。

对每一组和每种尺度计算 Pearson、Spearman、`mean(d)`、`mean(|d|)`、`|d|` 的 P50/P90/P99、`|d|<1e-6` 占比和完全相等占比。另列出两模型同时落在相同裁剪边界的占比，并核对原始输出重新换算后的 `p_hat` 与第 8 节保存的逐行预测是否一致。统计和差异直方图使用**全部 Test 行**；散点图固定抽取 20,000 行。运行本节需要原 kernel 中保留第 8 节的模型与 `test_predictions`。

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

diag_required = [
    "test_predictions", "records", "splits", "RF_BATCH", "MIN_P", "MAX_P",
    "LOG_MIN_H", "LOG_MAX_H", "rf_x_from_records", "rf_p_x_from_records",
    "rf_e1_model", "rf_e2_model", "lgbm_e3_model", "lgbm_e4_model",
    "spearman_with_ties",
]
diag_missing = [name for name in diag_required if name not in globals()]
if diag_missing:
    raise RuntimeError(
        "Missing variables from the current kernel: " + ", ".join(diag_missing)
        + ". This section needs the trained models and section 8 test_predictions."
    )

diag_test_start, diag_test_stop = splits["Test"]
diag_n_test = diag_test_stop - diag_test_start

def diag_metric_row(pair, space, rf_values, lgb_values):
    rf_values = np.asarray(rf_values, dtype=np.float64)
    lgb_values = np.asarray(lgb_values, dtype=np.float64)
    assert rf_values.shape == lgb_values.shape == (diag_n_test,)
    assert np.isfinite(rf_values).all() and np.isfinite(lgb_values).all()
    d = rf_values - lgb_values
    abs_d = np.abs(d)
    p50, p90, p99 = np.quantile(abs_d, [0.50, 0.90, 0.99])
    pearson = (float(np.corrcoef(rf_values, lgb_values)[0, 1])
               if np.std(rf_values) > 0 and np.std(lgb_values) > 0
               else np.nan)
    return {
        "pair": pair, "space": space, "n_test": diag_n_test,
        "Pearson": pearson,
        "Spearman": spearman_with_ties(rf_values, lgb_values),
        "mean_d": float(d.mean()),
        "mean_abs_d": float(abs_d.mean()),
        "P50_abs_d": float(p50), "P90_abs_d": float(p90),
        "P99_abs_d": float(p99),
        "share_abs_lt_1e-6": float(np.mean(abs_d < 1e-6)),
        "share_exact": float(np.mean(d == 0)),
    }

diag_pairs = [
    ("E1 vs E3", "RF-logh (E1)", "LGBM-logh (E3)",
     rf_e1_model, lgbm_e3_model, rf_x_from_records, "raw_log2_h", True),
    ("E2 vs E4", "RF-p (E2)", "LGBM-p (E4)",
     rf_e2_model, lgbm_e4_model, rf_p_x_from_records, "raw_p", False),
]
diag_stat_rows = []
diag_boundary_rows = []
diag_check_rows = []

for pair, rf_name, lgb_name, rf_model, lgb_model, make_x, raw_space, predicts_logh in diag_pairs:
    rf_p = np.asarray(test_predictions[rf_name][0], dtype=np.float64)
    lgb_p = np.asarray(test_predictions[lgb_name][0], dtype=np.float64)
    assert rf_p.shape == lgb_p.shape == (diag_n_test,)
    diag_stat_rows.append(diag_metric_row(pair, "clipped_p", rf_p, lgb_p))

    same_bound = ((rf_p == MIN_P) & (lgb_p == MIN_P)) | (
        (rf_p == MAX_P) & (lgb_p == MAX_P)
    )
    both_interior = ((MIN_P < rf_p) & (rf_p < MAX_P)
                     & (MIN_P < lgb_p) & (lgb_p < MAX_P))
    diag_boundary_rows.append({
        "pair": pair,
        "same_p_bound_share": float(same_bound.mean()),
        "n_both_interior": int(both_interior.sum()),
        "share_abs_lt_1e-6_interior": (
            float(np.mean(np.abs(rf_p[both_interior] - lgb_p[both_interior]) < 1e-6))
            if both_interior.any() else np.nan
        ),
    })

    rf_raw = np.empty(diag_n_test, dtype=np.float64)
    lgb_raw = np.empty(diag_n_test, dtype=np.float64)
    rf_reconstruction_max = 0.0
    lgb_reconstruction_max = 0.0
    for left in range(diag_test_start, diag_test_stop, RF_BATCH):
        right = min(left + RF_BATCH, diag_test_stop)
        batch = records[left:right]
        x = make_x(batch)
        lo, hi = left - diag_test_start, right - diag_test_start
        rf_chunk = rf_model.predict(x)
        lgb_chunk = lgb_model.predict(x, num_iteration=lgb_model.best_iteration_)
        rf_raw[lo:hi] = rf_chunk
        lgb_raw[lo:hi] = lgb_chunk

        if predicts_logh:
            rf_h = np.exp2(np.clip(rf_chunk, LOG_MIN_H, LOG_MAX_H))
            lgb_h = np.exp2(np.clip(lgb_chunk, LOG_MIN_H, LOG_MAX_H))
            rf_rebuilt = np.clip(
                np.exp2(-batch["delta_days"] / rf_h), MIN_P, MAX_P
            )
            lgb_rebuilt = np.clip(
                np.exp2(-batch["delta_days"] / lgb_h), MIN_P, MAX_P
            )
        else:
            rf_rebuilt = np.clip(rf_chunk, MIN_P, MAX_P)
            lgb_rebuilt = np.clip(lgb_chunk, MIN_P, MAX_P)
        rf_reconstruction_max = max(
            rf_reconstruction_max, float(np.max(np.abs(rf_rebuilt - rf_p[lo:hi])))
        )
        lgb_reconstruction_max = max(
            lgb_reconstruction_max, float(np.max(np.abs(lgb_rebuilt - lgb_p[lo:hi])))
        )

    diag_stat_rows.append(diag_metric_row(pair, raw_space, rf_raw, lgb_raw))
    diag_check_rows.append({
        "pair": pair,
        "max_rebuild_error_RF": rf_reconstruction_max,
        "max_rebuild_error_LGBM": lgb_reconstruction_max,
    })
    del rf_raw, lgb_raw, same_bound, both_interior

diag_stats = pd.DataFrame(diag_stat_rows)
diag_boundary = pd.DataFrame(diag_boundary_rows)
diag_reconstruction = pd.DataFrame(diag_check_rows)
assert (diag_reconstruction[["max_rebuild_error_RF",
                             "max_rebuild_error_LGBM"]].to_numpy() < 1e-10).all()
with pd.option_context("display.float_format", lambda value: f"{value:.6g}"):
    display(diag_stats)
    display(diag_boundary)
    display(diag_reconstruction)


In [ ]:
import matplotlib.pyplot as plt

diag_scatter_n = min(20_000, diag_n_test)
diag_sample = np.random.default_rng(20261115).choice(
    diag_n_test, size=diag_scatter_n, replace=False
)
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
for row, (pair, rf_name, lgb_name, *_rest) in enumerate(diag_pairs):
    rf_p = test_predictions[rf_name][0]
    lgb_p = test_predictions[lgb_name][0]
    ax_scatter, ax_hist = axes[row]
    ax_scatter.scatter(
        rf_p[diag_sample], lgb_p[diag_sample],
        s=7, alpha=0.18, linewidths=0, rasterized=True,
    )
    ax_scatter.plot([MIN_P, MAX_P], [MIN_P, MAX_P], "k--", linewidth=1)
    ax_scatter.set(xlim=(0, 1), ylim=(0, 1),
                   xlabel="RF p_hat", ylabel="LightGBM p_hat",
                   title=f"{pair}: sampled Test predictions (n={diag_scatter_n:,})")
    ax_scatter.set_aspect("equal", adjustable="box")
    ax_scatter.grid(alpha=0.2)

    abs_d = np.abs(rf_p - lgb_p)
    positive = abs_d[abs_d > 0]
    ax_hist.hist(np.maximum(positive, 1e-12),
                 bins=np.logspace(-12, 0, 75),
                 color="#4C78A8", edgecolor="none")
    ax_hist.set_xscale("log")
    ax_hist.set(xlim=(1e-12, 1), xlabel="|p_RF - p_LightGBM|",
                ylabel="Test row count",
                title=f"{pair}: all Test rows; exact zero = {(abs_d == 0).mean():.2%}")
    ax_hist.grid(alpha=0.2)

fig.suptitle("RF vs LightGBM: paired prediction diagnostics", fontsize=15)
fig.tight_layout()
plt.show()


**读表与读图：**相关系数接近 1 表示排序或线性关系接近，不能单独证明逐条预测相同；要同时看绝对差异及裁剪前结果。`share_*` 列为 0–1 的比例。散点图中虚线是 `y=x`；右侧直方图统计全部 Test 行的正差异，横轴为对数刻度，完全相等的占比写在标题中。小于 `1e-12` 的正差异归入第一个柱。

## 11.2 树模型更适合排序，HLR 更适合概率预测吗？

本节对同一批 Test session 比较 HLR、N-HLR、RF-p (E2)、LGBM-p (E4)，复用第 8 节的 `test_predictions`，不重新训练。实验 A 检查记录级预测 `p_hat` 对裁剪后 `p_target` 的校准；实验 B 检查预测最低的前 K% 记录是否真的有较高的 `1-p_target`。Train 均值常数预测用于校准参照；随机排序的期望 Lift 为 1。

**指标口径：**每条 session 等权。ECE 使用 10 个等宽预测区间，并同时展示每箱行数；MSE 用于补充校准与预测区分度。AUC 沿用第 8 节的 `np.rint(p_target)` 二值标签，Risk@K 则使用连续的 `1-p_target`。二者衡量的排序目标不同。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

task93_needed = [
    "records", "train_end", "test_true_p", "test_days",
    "test_predictions", "auc_fixed", "MIN_P", "MAX_P",
]
task93_missing = [name for name in task93_needed if name not in globals()]
if task93_missing:
    raise RuntimeError(
        "Section 11.2 needs the current kernel's section 8 results: "
        + ", ".join(task93_missing)
    )

task93_model_names = ["HLR", "N-HLR", "RF-p (E2)", "LGBM-p (E4)"]
task93_true = np.asarray(test_true_p, dtype=np.float64)
task93_days = np.asarray(test_days, dtype=np.float64)
task93_n = len(task93_true)
task93_train_mean = float(records["p_target"][:train_end].mean())
task93_predictions = {
    name: np.asarray(test_predictions[name][0], dtype=np.float64)
    for name in task93_model_names
}
task93_predictions["Train mean"] = np.full(
    task93_n, task93_train_mean, dtype=np.float64
)
assert len(task93_days) == task93_n
assert all(len(pred) == task93_n and np.isfinite(pred).all()
           for pred in task93_predictions.values())
assert all(np.all((MIN_P <= pred) & (pred <= MAX_P))
           for pred in task93_predictions.values())

task93_edges = np.linspace(0.0, 1.0, 11)
task93_bin_labels = [
    f"[{task93_edges[b]:.1f}, {task93_edges[b+1]:.1f})"
    if b < 9 else "[0.9, 1.0]"
    for b in range(10)
]
task93_binary_recall = np.rint(task93_true)
task93_cal_rows = []
task93_summary_rows = []
for name, pred in task93_predictions.items():
    bin_id = np.clip(
        np.searchsorted(task93_edges, pred, side="right") - 1, 0, 9
    )
    ece = 0.0
    for b, label in enumerate(task93_bin_labels):
        mask = bin_id == b
        n_bin = int(mask.sum())
        mean_pred = float(pred[mask].mean()) if n_bin else np.nan
        mean_true = float(task93_true[mask].mean()) if n_bin else np.nan
        gap = mean_true - mean_pred if n_bin else np.nan
        if n_bin:
            ece += (n_bin / task93_n) * abs(gap)
        task93_cal_rows.append({
            "model": name, "pred_bin": label, "n_bin": n_bin,
            "share": n_bin / task93_n,
            "mean_pred": mean_pred, "mean_true": mean_true,
            "calibration_gap_true_minus_pred": gap,
        })
    error = pred - task93_true
    mse = float(np.mean(error ** 2))
    task93_summary_rows.append({
        "model": name, "n_test": task93_n,
        "mean_pred": float(pred.mean()), "std_pred": float(pred.std()),
        "MAE": float(np.mean(np.abs(error))),
        "MSE": mse, "RMSE": float(np.sqrt(mse)),
        "ECE10": float(ece),
        "AUC": auc_fixed(task93_binary_recall, pred),
    })

task93_calibration = pd.DataFrame(task93_cal_rows)
task93_summary = pd.DataFrame(task93_summary_rows)
task93_counts = task93_calibration.pivot(
    index="pred_bin", columns="model", values="n_bin"
).loc[task93_bin_labels, list(task93_predictions)]
print(f"Test mean p_target = {task93_true.mean():.6f}; "
      f"Train mean constant = {task93_train_mean:.6f}")
display(task93_summary.round(6))
display(task93_counts)
display(task93_calibration.round(6))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ["#4C78A8", "#54A24B", "#F58518", "#E45756", "#777777"]
axes[0].plot([0, 1], [0, 1], "k--", linewidth=1, label="perfect calibration")
for (name, color) in zip(task93_predictions, colors):
    part = task93_calibration[
        (task93_calibration["model"] == name)
        & (task93_calibration["n_bin"] > 0)
    ]
    axes[0].plot(
        part["mean_pred"], part["mean_true"],
        color=color, linewidth=1.3, alpha=0.8,
    )
    axes[0].scatter(
        part["mean_pred"], part["mean_true"],
        s=24 + 180 * np.sqrt(part["share"]),
        color=color, alpha=0.85, label=name,
    )
    occupied = part["pred_bin"].map(
        {label: i for i, label in enumerate(task93_bin_labels)}
    ).to_numpy()
    axes[1].plot(
        occupied + 0.5, part["n_bin"], marker="o",
        color=color, linewidth=1.3, label=name,
    )
axes[0].set(
    title="Reliability diagram (marker size reflects bin share)",
    xlim=(0, 1), ylim=(0, 1),
    xlabel="mean predicted p in bin",
    ylabel="mean observed p_target in bin",
)
axes[1].set(
    title="Test rows in each prediction bin",
    xticks=np.arange(10) + 0.5,
    xticklabels=[f"{b/10:.1f}" for b in range(10)],
    xlabel="predicted p bin (left edge)", ylabel="n_bin",
)
axes[1].set_yscale("log")
for ax in axes:
    ax.grid(alpha=0.2)
axes[0].legend(fontsize=8, loc="upper left")
fig.tight_layout()
plt.show()


### 实验 B：Top-K 回忆失败风险排序

按预测 `p_hat` 从低到高排序，取前 5%、10%、20% 的 Test 行，计算其真实平均风险 `Risk@K=mean(1-p_target)` 和 `Lift@K=Risk@K/mean(1-p_target)`。曲线展示 1%–50%。随机排序基线只给出**期望值**（Risk 等于总体平均风险、Lift=1）；常数预测存在完全并列，不能产生有意义的排序。

另在 `[0,3)`、`[3,7)`、`[7,30)`、`[30,+∞)` 天四个 delta 区间内分别取最低预测的前 10%，以检查总体排序优势是否仅来自复习间隔差异。

In [ ]:
task93_risk = 1.0 - task93_true
task93_base_risk = float(task93_risk.mean())
assert task93_base_risk > 0
task93_curve_rows = []
task93_selected_rows = []
task93_curve_percents = np.arange(1, 51)

for name in task93_model_names:
    order = np.argsort(task93_predictions[name], kind="stable")
    cumulative_risk = np.cumsum(task93_risk[order], dtype=np.float64)
    for percent in task93_curve_percents:
        k = int(np.ceil(task93_n * percent / 100))
        selected_risk = float(cumulative_risk[k - 1] / k)
        row = {
            "model": name, "K_percent": int(percent), "n_selected": k,
            "Risk_at_K": selected_risk,
            "Lift_at_K": selected_risk / task93_base_risk,
        }
        task93_curve_rows.append(row)
        if percent in (5, 10, 20):
            task93_selected_rows.append(row)

for percent in (5, 10, 20):
    task93_selected_rows.append({
        "model": "Random expected", "K_percent": percent,
        "n_selected": int(np.ceil(task93_n * percent / 100)),
        "Risk_at_K": task93_base_risk, "Lift_at_K": 1.0,
    })
task93_curve = pd.DataFrame(task93_curve_rows)
task93_selected = pd.DataFrame(task93_selected_rows)
print(f"Overall Test forgetting risk = {task93_base_risk:.6f}")
display(task93_selected.round(6))

task93_joint_rows = []
for _, summary_row in task93_summary.iterrows():
    name = summary_row["model"]
    row = {key: summary_row[key] for key in
           ("model", "MAE", "MSE", "ECE10", "AUC")}
    for percent in (10, 20):
        if name == "Train mean":
            risk_k, lift_k = task93_base_risk, 1.0
        else:
            selected = task93_selected.loc[
                (task93_selected["model"] == name)
                & (task93_selected["K_percent"] == percent)
            ].iloc[0]
            risk_k = float(selected["Risk_at_K"])
            lift_k = float(selected["Lift_at_K"])
        row[f"Risk@{percent}%"] = risk_k
        row[f"Lift@{percent}%"] = lift_k
    task93_joint_rows.append(row)
task93_joint = pd.DataFrame(task93_joint_rows)
print("Train mean has no ranking; its Risk/Lift columns show random-tie expectation.")
display(task93_joint.round(6))

fig, ax = plt.subplots(figsize=(9, 5))
colors = ["#4C78A8", "#54A24B", "#F58518", "#E45756"]
for name, color in zip(task93_model_names, colors):
    part = task93_curve[task93_curve["model"] == name]
    ax.plot(
        part["K_percent"], part["Lift_at_K"],
        linewidth=2, label=name, color=color,
    )
ax.axhline(1.0, color="black", linestyle="--", linewidth=1,
           label="random expected")
ax.set(
    title="Top-K forgetting-risk lift on Test",
    xlabel="lowest predicted p selected (%)",
    ylabel="Lift@K", xlim=(1, 50),
)
ax.grid(alpha=0.2)
ax.legend()
fig.tight_layout()
plt.show()

task93_delta_names = ["[0, 3) days", "[3, 7) days",
                      "[7, 30) days", "[30, +inf) days"]
task93_delta_id = np.searchsorted(
    np.array([3.0, 7.0, 30.0]), task93_days, side="right"
)
task93_delta_rows = []
for band, band_name in enumerate(task93_delta_names):
    indices = np.flatnonzero(task93_delta_id == band)
    n_band = len(indices)
    base_band_risk = float(task93_risk[indices].mean())
    k = int(np.ceil(n_band * 0.10))
    for name in task93_model_names:
        order = np.argsort(task93_predictions[name][indices], kind="stable")
        selected = indices[order[:k]]
        selected_risk = float(task93_risk[selected].mean())
        task93_delta_rows.append({
            "delta_band": band_name, "model": name, "n_band": n_band,
            "n_selected": k, "band_mean_risk": base_band_risk,
            "Risk@10%": selected_risk,
            "Lift@10%": selected_risk / base_band_risk,
        })
task93_delta_ranking = pd.DataFrame(task93_delta_rows)
display(task93_delta_ranking.round(6))
